# ABI-S15 Write Up Attempt

## Data & Embedding - Foundations

To explore whether conversational convergence and divergence can be computationally detected in multimodal team conversations, I began with ABI_S15 as a pilot case. This session was selected because it contains multiple decision points and substantive technical discussion, making it a strong testbed for validating whether automated methods can capture meaningful group dynamics. The transcript was segmented into sentences-level units, which served as the smallest analytic granularity. Each row of the resulting dataset contained the text of the sentence, its speaker identifier, and a timestamp (in seconds). Codebook annotations - such as *decision*, *explanation*, and *new idea* - were aligned at the sentence or phrase level, allowing me to later compare computationally detected convergence with human-coded markers.

Each sentence $s_i$ was then mapped into a dense vector embedding using a pretrained transformer model (SentenceTransformer, `all-MiniLM-L6-v2`). Formally, the transcript is represented as a sequence

$D = \{(s_i, t_i, u_i)\}^{N}_{i=1}$

Where $s_i$ is the sentence text, $t_i$ the timestamp, and $u_i$ the speaker. The embedding function $f(\cdot)$ maps each sentence into a $d$-dimensional semantic space:

$e_i = f(s_i), e_i \in \mathbb{R}^d$

In practice, $d = 384$ for MiniLM. For interpretability and numerical stability, all embeddings were $L_2$-normalized, giving unit vectors $\tilde{e_i}$.

Semantic similarity between any two sentences is then defined by cosine similarity:

$sim(\tilde{e_i}, \tilde{e_j} = \tilde{e_i}^T\tilde{e_j})$

which reduces to the dot product of the normalized embeddings. Scores closer to 1 indicate high semantic overlap, while lower scores reflect divergence. This embedding space establishes the foundation for detecting moments when participants independently produce semantically similar contributions in close temporal proximity - our operational definition of conversational convergence.

## Convergence Detection via Time-Bounded Bursts

Having established a semantic similarity measure at the sentence level, the next step is to detect *bursts of convergence* - intervals where multiple  speakers independently articulate semantically similar content within a short period of time.

To operationalize this, I defined **temporal windows** centered at either *(a)* fixed increments (sliding windows) or *(b)* pre-coded "decision-like" lines. For each window $W_k$, defined by a center $\tau_k$ and a half-width $\Delta$, the set of sentences included is:

$W_k = \{i | t_i - \tau_k \leq \Delta\}$

Where $t_i$ denotes the timestamp of sentence $i$. In this analysis, $\Delta = 45$ seconds.

Within each window, I considered all cross-speaker pairs of sentences that occur within a smaller temporal gap $\delta$, in order to capture near-simultaneous contributions rather than statement far apart in time. The set of candidate pairs is:

$\mathcal{P}_k = \{(i,j) \in W_k \times W_k | i< j, | t_i - t_j | \leq \delta, u_i \neq u_j\},$

where $u_i$ denotes the speaker identity.

For each window, I then computed a **burst score** that quantifies the proportion of cross-speakers that exceeded a similarity threshold $\theta$:

$B(W_k) = \frac{1}{|\mathcal{P}_k|}\sum_{(i,j) \in \mathcal{P}_k}1[\tilde{e_i}^T\tilde{e_j} \geq \theta],$

where $1[\cdot]$ is the indicator function. Intuitively, $B(W_k)$ captures the density of semantically overlapping contributions in the window.

A window $W_k$ is flagged as a **convergence burst** if two conditions are met:

1. The burst score exceeds a threshold $(B(W_k) \geq \beta)$ and 

2. Contributions come from at least $m$ distinct speakers

For ABI_S15, thresholds were selected empirically $(\theta \approx 0.75, \beta \approx 0.20, m = 3)$. Flagged windows (e.g. the 913-918s segment) represent candidate convergence moments, which can then be validated against transcript content and codebook labels.

## Overlay with Codebook Annotations and Topic Terms

To interpret the candidate convergence burst identified in Section 2, I overlaid them with the human-coded annotations and key topical terms present in the transcript. The goal was to determine whether computationally flagged windows corresponded to moments of substantive group alignment, as reflected in the coding scheme and content of the discussion.

Each sentence in the transcript already carried one or more codebook labels, such as *decision*, *explanation*, or *new idea*. These labels provide an independent benchmark for assessing whether bursts correspond to meaningful conversational events. For each flagged window $W_k$, I identified the subset of annotations within that timespan:

$C(W_k) = \{c_i | i \in W_k, c_i \in \text{Codebook Labels}\}$

If $\mathcal{C}(W_k)$ contained one or more decision-related codes, this provided preliminary evidence that the burst aligned with genuine group decision-making. Conversely, if the window contained only *explanation* or *new idea* labels, the burst might represent exploratory convergence without closure.

In addition to the annotations, I extracted **topic terms** from the transcript within each flagged window. These were domain-specific keywords (e.g. "photoacoustic", "two-photon") that ground the conversation in technical content. Formally the set of topic terms is defined as:

$T(W_k) = \{\, w \in \mathcal{V} \;|\; \exists i \in W_k \text{ such that } w \subset s_i\,\}$


Where $\mathcal{V}$ is a curated vocabulary of technical terms. These topics terms serve two roles:

a. they provide semantic anchors for interpreting the burst.

b. they allow convergence patterns to be tied to specific scientific concepts.

These three parts creates the layered representation of ABI_S15, allowing inspection of whether semantically similar contributions were clustering around points of explanation.

### An example

In the ~14-15 minute region, the flagged window coincided with both the label *decision* and topical mentions of "photoacoustic" and "two-photon". This strengthens the case that computationally detected bursts can identify moments of substantive convergence.

## Transcript Validation and Examples

While the overlay convergence bursts with codebook annotations and topic terms provides a useful interpretive scaffold, it does not guarantee that all flagged windows represent the true convergence. To assess validity, I manually reviewed transcript excerpts corresponding to flagged windows in ABI_S15. This step ensured that bursts reflected substantive conversational alignment rather than superficial similarity.

Formally, for each flagged window $W_k$, I retrieved the sequence of raw sentences 
$\{\, s_i : i \in W_k \,\}$ and inspected whether the content indicated:

a. agreement, reinforcement, or coordinated reasoning across multiple speakers (convergence), or

b. divergence, tangent, or exploratory talk without closure. This process is expressed as:

$V(W_k) = \text{Qualitative Judgement}(\{s_i : i \in W_k\})$,

where $V(W_k)$ is a binary or categorical outcome (e.g., "confirmed convergence", "exploratory but not convergent", "false positive")

This validation step also showed strengths and limitations of the method. For example, the window spanning 913-918s was flagged as a convergence burst because several speakers made semantically similar contributions within a short span.  However, upon transcript review, the discussion did not culminate to a final decision; instead, it represented exploratory repetition without resolution. By contrast, in the 14-15 minute region, a flagged burst coincided with explicit *decision codes* and topical discussion of "photoacoustic" and "two-photon" methods. Transcript review confirmed that the group was aligning around a technical solution, validating the burst as genuine convergence.

Computational detection can reliably identify where semantically dense clusters occur given enough training, but human inspection remains necessary to distinguish substantive convergence (shared decisions or aligned reasoning) from surface-level similarity (e.g., repeated terms without agreement).

## What comes next

The method I tried - spotting when people say very similar things close together in time - does find real "bursts" in the conversation. Some of those bursts line up with the actual decisions in the transcript (like around the 14-15 minutes, where the team chose between methods). Others look like bursts but don't actually lead anywhere (like 913-918s, where people repeated ideas but didn't reach agreement).

One would still have to check against the transcript to know whether a burst is real convergence or just noise.